In [1]:
# import os
# os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [3]:
transformers.__version__

'4.36.0'

In [6]:
base_model_id = "meta-llama/Llama-2-7b-chat-hf"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=getattr(torch, "float16"),
    bnb_4bit_use_double_quant=False,
)

base_model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, add_bos_token=True, trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
torch.__version__

'2.1.1+cu118'

In [9]:
from peft import PeftModel
rel_path = "train/mistral-nollama2-7b-chat/checkpoint-1100"
base_model = PeftModel.from_pretrained(base_model, f"../{rel_path}")

In [15]:
from transformers.generation import GenerationConfig

def make_prompt(inst):
    return f"""<s>[INST] {inst} [/INST]"""

Q = make_prompt("Skriv meg et dikt som handler om jula")
print(Q)
model_input = tokenizer(Q, return_tensors="pt")
base_model.eval();
with torch.no_grad():
    gen = tokenizer.decode(
        base_model.generate(**model_input, max_new_tokens=200, repetition_penalty=1.15)[0],
        skip_special_tokens=True
    )
    print(gen)


<s>[INST] Skriv meg et dikt som handler om jula [/INST]


/data/users/tollefj/venv/lib/python3.10/site-packages/transformers/generation/utils.py:1636: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


[INST] Skriv meg et dikt som handler om jula [/INST] Julelys klarer himmelen,
 
En kort tid på jordens bane.
Nattfugler synes foreldreløse, 
Sjøen blir snart fylt med svaner.
Etter hvert når kvelden inn,
Vi samles rundt en lysrein.
Julen er vår stund til glede og fest,
Og vi vil feire den med alle krefter. 
Lyser og rødt stråler lukker i,
Hjemlengseln fyller oss med glede. 
Farvinnede figurer går ut av skapet,
Gudskjenninger som lærer oss om kjærligheten.
Hver gang vi ser dem, de smiler på oss,
Til tross
